In [ ]:
import numpy as np, pandas as pd, requests
import glob, io, os, re, shutil, time, zipfile, json
import pyarrow as pa, pyarrow.parquet as pq
from pathlib import Path
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

BASE_DIR   = Path("") 

RAW_DIR = BASE_DIR / "Data" / "Raw"
RAW_DIR.mkdir(parents=True, exist_ok=True) 

INTERIM_DIR = BASE_DIR / "Data" / "Interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True) 

PROCESSED_DIR = BASE_DIR / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True) 

MERGED_DIR = BASE_DIR / "Data" / "Merged" 
MERGED_DIR.mkdir(parents=True, exist_ok=True) 

FINAL_DIR = BASE_DIR / "Data" / "Final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Epraccur

## Download

In [3]:
URL = "https://www.odsdatasearchandexport.nhs.uk/api/getReport?report=epraccur"

csv_path = RAW_DIR / "epraccur.csv"
part_path = RAW_DIR / "epraccur.csv.part"

# 1. Download (to .part first, then rename)
with requests.get(URL, stream=True, timeout=120, headers={"User-Agent": "Mozilla/5.0"}) as r:
    r.raise_for_status()
    with open(part_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
part_path.replace(csv_path)
print(f"Saved to {csv_path} ({csv_path.stat().st_size / 1e6:,.1f} MB)")

# 2. Quick look. Check whether this version has a header row.
epraccur = pd.read_csv(csv_path, header=None, dtype=str)
print(epraccur.shape)
display(epraccur.head())

Saved to /Users/trinityrose/Desktop/TEST/Data/Raw/epraccur.csv (3.3 MB)
(15692, 27)


,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
0,A81001,THE DENSHAM SURGERY,Y63,QHM,HEALTH CENTRE,LAWSON STREET,NaN,STOCKTON-ON-TEES,NaN,TS18 1HU,...,01642 672351,NaN,NaN,NaN,1,NaN,16C,NaN,RO76,NaN
1,A81002,QUEENS PARK MEDICAL CENTRE,Y63,QHM,FARRER STREET,NaN,NaN,STOCKTON ON TEES,NaN,TS18 2AW,...,01642 618170,NaN,NaN,NaN,1,NaN,16C,NaN,RO76,NaN
2,A81003,VICTORIA MEDICAL PRACTICE,Y63,QHM,HEALTH CENTRE,VICTORIA ROAD,NaN,HARTLEPOOL,NaN,TS26 8DB,...,01429 272945,NaN,NaN,NaN,1,NaN,00K,NaN,RO76,NaN
3,A81004,ACKLAM MEDICAL CENTRE,Y63,QHM,TRIMDON AVENUE,ACKLAM,NaN,MIDDLESBROUGH,NaN,TS5 8SB,...,01642 827697,NaN,NaN,NaN,1,NaN,16C,NaN,RO76,NaN
4,A81005,SPRINGWOOD SURGERY,Y63,QHM,RECTORY LANE,NaN,NaN,GUISBOROUGH,NaN,TS14 7DJ,...,01287 619611,NaN,NaN,NaN,1,NaN,16C,NaN,RO76,NaN


## Process

In [4]:
# epraccur has no header row: map file position -> name, keep only what's needed
EPRAC_COLS = {
    0:  "PRACTICE_CODE",
    1:  "PRACTICE_NAME",
    3:  "ICB_CODE",
    9:  "POSTCODE",
    10: "OPEN_DATE",
    11: "CLOSE_DATE",
    12: "STATUS",
    14: "COMMISSIONER",      # Sub ICB Location code
    25: "PRESCRIBING_SETTING",
}

# Study window (used later for filtering)
START_WINDOW = pd.Timestamp("2022-07-01")   # ICB introduction
END_WINDOW   = pd.Timestamp("2026-03-31")

eprac = (epraccur
      .rename(columns=EPRAC_COLS)
      [list(EPRAC_COLS.values())]
)

In [5]:
# ---- 1. Parse YYYYMMDD text to datetime ----
for col in ["OPEN_DATE", "CLOSE_DATE"]:
    eprac[col] = pd.to_datetime(eprac[col], format="%Y%m%d", errors="coerce")

print(eprac[["OPEN_DATE", "CLOSE_DATE"]].isna().sum())
print(eprac["OPEN_DATE"].min(), eprac["OPEN_DATE"].max())

# ---- 2. England only: ICB_CODE is mapped from the practice postcode to NHS England geography ----
n_all = eprac["PRACTICE_CODE"].nunique()
eng = eprac.loc[eprac["ICB_CODE"].notna()].copy()
print(f"Excluded (no ICB code, i.e. outside English health geography): "
      f"{n_all - eng['PRACTICE_CODE'].nunique():,} of {n_all:,} practice codes")

# ---- 3. Keep practices active at any point in the study window ----
active = eng.loc[
    (eng["OPEN_DATE"] <= END_WINDOW)
    & (eng["CLOSE_DATE"].isna() | (eng["CLOSE_DATE"] >= START_WINDOW))
].copy()

# ---- 4. Duplicate practice codes ----
dupes = active["PRACTICE_CODE"].duplicated(keep=False).sum()
print(f"Rows with a repeated PRACTICE_CODE: {dupes}")

# ---- 5. Inspect the prescribing settings present ----
setting_counts = (
    active.groupby("PRESCRIBING_SETTING", dropna=False)["PRACTICE_CODE"]
    .nunique()
    .sort_values(ascending=False)
)
display(setting_counts)

# ---- 6. Filter to GP practices (RO76, including RO76|RO268) ----
# Exact token match, so RO76 is not matched inside a longer code
in_scope = active.loc[
    active["PRESCRIBING_SETTING"].str.contains(r"(?:^|\|)RO76(?:\||$)", na=False)
].copy()

# ---- 7. Verify ----
print(f"Rows remaining: {len(in_scope):,}")
print(f"Unique practices remaining: {in_scope['PRACTICE_CODE'].nunique():,}")
print("\nBreakdown by setting:")
print(in_scope["PRESCRIBING_SETTING"].value_counts())

OPEN_DATE         0
CLOSE_DATE    12849
dtype: int64
1974-04-01 00:00:00 2026-11-01 00:00:00
Excluded (no ICB code, i.e. outside English health geography): 748 of 15,692 practice codes
Rows with a repeated PRACTICE_CODE: 0


PRESCRIBING_SETTING
RO76          6633
RO247         1763
RO72          1442
RO250          802
RO321          623
RO80           273
RO255          265
RO252          232
RO323          224
RO259          161
RO249          153
RO82           127
RO87            83
RO246           72
RO256           18
RO80|RO87       17
RO251           10
RO76|RO268       6
RO258            5
RO260            4
RO257            2
RO248            2
NaN              1
Name: PRACTICE_CODE, dtype: int64

Rows remaining: 6,639
Unique practices remaining: 6,639

Breakdown by setting:
PRESCRIBING_SETTING
RO76          6633
RO76|RO268       6
Name: count, dtype: int64


In [6]:
out_file = PROCESSED_DIR / "epraccur_processed.parquet"

# Convert the nullable string columns to plain strings so parquet round-trips cleanly
save_df = in_scope.copy()
for c in save_df.columns:
    if pd.api.types.is_string_dtype(save_df[c]):
        save_df[c] = save_df[c].astype("object")

save_df.to_parquet(out_file, index=False)

# Quick check by reading it back
chk = pd.read_parquet(out_file)
print(f"Saved {len(chk):,} rows, {chk.shape[1]} columns to {out_file} ({out_file.stat().st_size / 1e6:,.2f} MB)")
print(f"Unique practices: {chk['PRACTICE_CODE'].nunique():,}")
print(chk.dtypes)

Saved 6,639 rows, 9 columns to /Users/trinityrose/Desktop/TEST/Data/Processed/epraccur_processed.parquet (0.20 MB)
Unique practices: 6,639
PRACTICE_CODE                  object
PRACTICE_NAME                  object
ICB_CODE                       object
POSTCODE                       object
OPEN_DATE              datetime64[ns]
CLOSE_DATE             datetime64[ns]
STATUS                         object
COMMISSIONER                   object
PRESCRIBING_SETTING            object
dtype: object


# BNF

## Download

In [7]:
DATASET_ID = "bnf-code-information-current-year"
RESOURCE_ID = "5a9a1a67-0758-48b8-bcc9-0bcc3d0b61b9"
API_URL = f"https://opendata.nhsbsa.net/api/3/action/package_show?id={DATASET_ID}"

# 1. Find the resource and its download URL
resources = requests.get(API_URL, timeout=60).json()["result"]["resources"]
res = next(r for r in resources if r["id"] == RESOURCE_ID)
url = res["url"]
print("Name:", res.get("name"))
print("Format:", res.get("format"))
print("URL:", url)

# 2. Download (to .part first, then rename once complete)
ext = Path(url.split("?")[0]).suffix or ".csv"
final_path = RAW_DIR / f"bnf_code_information_current{ext}"
part_path = final_path.with_suffix(final_path.suffix + ".part")

with requests.get(url, stream=True, timeout=120) as r:
    r.raise_for_status()
    with open(part_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
part_path.replace(final_path)
print(f"Saved to {final_path} ({final_path.stat().st_size / 1e6:,.1f} MB)")

# 3. Quick look
bnf_raw = pd.read_csv(final_path, dtype=str)
print(bnf_raw.shape)
display(bnf_raw.head())

Name: BNF_CODE_CURRENT_202605_VERSION_90
Format: CSV
URL: https://opendata.nhsbsa.net/dataset/29d25de3-02cd-4755-9dee-cdc37e37b5f3/resource/5a9a1a67-0758-48b8-bcc9-0bcc3d0b61b9/download/bnf_code_current_202605_version_90.csv
Saved to /Users/trinityrose/Desktop/TEST/Data/Raw/bnf_code_information_current.csv (14.3 MB)
(55203, 15)


,YEAR_MONTH,BNF_CHAPTER,BNF_CHAPTER_CODE,BNF_SECTION,BNF_SECTION_CODE,BNF_PARAGRAPH,BNF_PARAGRAPH_CODE,BNF_SUBPARAGRAPH,BNF_SUBPARAGRAPH_CODE,BNF_CHEMICAL_SUBSTANCE,BNF_CHEMICAL_SUBSTANCE_CODE,BNF_PRODUCT,BNF_PRODUCT_CODE,BNF_PRESENTATION,BNF_PRESENTATION_CODE
0,2026-05,Gastro-Intestinal System,01,Dyspepsia and gastro-oesophageal reflux disease,0101,Antacids and simeticone,010101,Antacids and simeticone,0101010,Other antacid and simeticone preparations,010101000,Proprietary compound preparation BNF 0101010,010101000BB,Indigestion mixture,010101000BBAJA0
1,2026-05,Gastro-Intestinal System,01,Dyspepsia and gastro-oesophageal reflux disease,0101,Antacids and simeticone,010101,Antacids and simeticone,0101010,Alexitol sodium,0101010A0,Alexitol sodium,0101010A0AA,Alexitol sodium 360mg tablets,0101010A0AAAAAA
2,2026-05,Gastro-Intestinal System,01,Dyspepsia and gastro-oesophageal reflux disease,0101,Antacids and simeticone,010101,Antacids and simeticone,0101010,Alexitol sodium,0101010A0,Actal,0101010A0BB,Actal 360mg tablets,0101010A0BBAAAA
3,2026-05,Gastro-Intestinal System,01,Dyspepsia and gastro-oesophageal reflux disease,0101,Antacids and simeticone,010101,Antacids and simeticone,0101010,Aluminium hydroxide,0101010C0,Aluminium hydroxide,0101010C0AA,Aluminium hydroxide 475mg capsules,0101010C0AAAAAA
4,2026-05,Gastro-Intestinal System,01,Dyspepsia and gastro-oesophageal reflux disease,0101,Antacids and simeticone,010101,Antacids and simeticone,0101010,Aluminium hydroxide,0101010C0,Aluminium hydroxide,0101010C0AA,Aluminium hydroxide oral suspension sugar free,0101010C0AAACAC


## Process

In [8]:
bnf = pd.read_csv(RAW_DIR / "bnf_code_information_current.csv", dtype=str)
bnf_4 = bnf[bnf["BNF_PARAGRAPH_CODE"].astype(int).isin([
    40901, #Dopaminergic drugs used in parkinsonism
    40902, #Antimuscarinic drugs used in parkinsonism
    41100 #Dementia
    ])].copy()


DROP_CODES = ["0409010L0", "0409010P0"]   
bnf_4 = bnf_4[~bnf_4["BNF_CHEMICAL_SUBSTANCE_CODE"].str.strip().isin(DROP_CODES)].copy()

In [9]:
display(bnf_4['BNF_CHEMICAL_SUBSTANCE_CODE'].nunique())

bnf_4[["BNF_CHEMICAL_SUBSTANCE", "BNF_CHEMICAL_SUBSTANCE_CODE"]].drop_duplicates().sort_values("BNF_CHEMICAL_SUBSTANCE_CODE")

27

,BNF_CHEMICAL_SUBSTANCE,BNF_CHEMICAL_SUBSTANCE_CODE
9746,Apomorphine hydrochloride,0409010A0
9770,Safinamide,0409010AA
9774,Opicapone,0409010AB
9776,Apomorphine hydrochloride hemihydrate,0409010AC
9780,Amantadine hydrochloride,0409010B0
9785,Carbidopa,0409010D0
9786,Ropinirole hydrochloride,0409010H0
9840,Levodopa,0409010I0
9846,Co-beneldopa (Benserazide/levodopa),0409010K0
9862,Co-careldopa (Carbidopa/levodopa),0409010N0


In [10]:
bnf_4.to_csv(PROCESSED_DIR / "bnf_processed.csv")

# IMD

## Download

In [11]:
URL = ("https://assets.publishing.service.gov.uk/media/691ded56d140bbbaa59a2a7d/"
       "File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv")

csv_path = RAW_DIR / "IoD2025_File7_all_ranks_scores_deciles.csv"
part_path = csv_path.with_suffix(".csv.part")

# 1. Download (to .part first, then rename)
with requests.get(URL, stream=True, timeout=120, headers={"User-Agent": "Mozilla/5.0"}) as r:
    r.raise_for_status()
    with open(part_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
part_path.replace(csv_path)
print(f"Saved to {csv_path} ({csv_path.stat().st_size / 1e6:,.1f} MB)")

# 2. Quick look: print the column names so you can pick the right ones
imd_raw = pd.read_csv(csv_path, dtype=str)
print(imd_raw.shape)
print(imd_raw.columns.tolist())
display(imd_raw.head())

Saved to /Users/trinityrose/Desktop/TEST/Data/Raw/IoD2025_File7_all_ranks_scores_deciles.csv (9.9 MB)
(33755, 56)
['LSOA code (2021)', 'LSOA name (2021)', 'Local Authority District code (2024)', 'Local Authority District name (2024)', 'Index of Multiple Deprivation (IMD) Score', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)', 'Income Score (rate)', 'Income Rank (where 1 is most deprived)', 'Income Decile (where 1 is most deprived 10% of LSOAs)', 'Employment Score (rate)', 'Employment Rank (where 1 is most deprived)', 'Employment Decile (where 1 is most deprived 10% of LSOAs)', 'Education, Skills and Training Score', 'Education, Skills and Training Rank (where 1 is most deprived)', 'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)', 'Health Deprivation and Disability Score', 'Health Deprivation and Disability Rank (where 1 is most deprived)', 'Health 

,LSOA code (2021),LSOA name (2021),Local Authority District code (2024),Local Authority District name (2024),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Score,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2022,Dependent Children aged 0-15: mid 2022,Older population aged 60 and over: mid 2022,Working age population 18-66 (for use with Employment Deprivation Domain): mid 2022
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10,...,1.207,1105,1,1.414,1586,1,1795,149,520,1248
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10,...,0.355,9591,3,1.839,592,1,1671,81,387,1324
2,E01000003,City of London 001C,E09000001,City of London,9.25,25913,8,0.107,25167,8,...,0.318,10175,4,1.679,903,1,1896,136,432,1469
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5,...,0.012,15502,5,2.065,303,1,1737,177,160,1448
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3,...,0.399,8934,3,0.4,9136,3,1837,397,225,1260


## Process

In [12]:
for c in imd_raw.columns:
    print("  ", c)

# 1. Map raw column names to the ones you want (regex on the raw header)
PATTERNS = {
    "lsoa21cd":     r"^LSOA code",
    "lsoa21nm":     r"^LSOA name",
    "lad24cd":      r"^Local Authority District code",
    "lad24nm":      r"^Local Authority District name",
    "imd25_rank":   r"^Index of Multiple Deprivation.*Rank",
    "imd25_decile": r"^Index of Multiple Deprivation.*Decile",
}
rename = {}
for new, pat in PATTERNS.items():
    hits = [c for c in imd_raw.columns if re.search(pat, c, flags=re.I)]
    if len(hits) != 1:
        raise ValueError(f"{new}: expected 1 match for /{pat}/, found {hits}")
    rename[hits[0]] = new
print("\nMapping:")
for k, v in rename.items():
    print(f"  {v:14s} <- {k}")

imd = imd_raw[list(rename)].rename(columns=rename).copy()

# 2. Set datatypes
for c in ["lsoa21cd", "lsoa21nm", "lad24cd", "lad24nm"]:
    imd[c] = imd[c].astype("string").str.strip()
imd["imd25_rank"] = pd.to_numeric(imd["imd25_rank"], errors="coerce").astype("Int32")
imd["imd25_decile"] = pd.to_numeric(imd["imd25_decile"], errors="coerce").astype("Int8")

   LSOA code (2021)
   LSOA name (2021)
   Local Authority District code (2024)
   Local Authority District name (2024)
   Index of Multiple Deprivation (IMD) Score
   Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)
   Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)
   Income Score (rate)
   Income Rank (where 1 is most deprived)
   Income Decile (where 1 is most deprived 10% of LSOAs)
   Employment Score (rate)
   Employment Rank (where 1 is most deprived)
   Employment Decile (where 1 is most deprived 10% of LSOAs)
   Education, Skills and Training Score
   Education, Skills and Training Rank (where 1 is most deprived)
   Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)
   Health Deprivation and Disability Score
   Health Deprivation and Disability Rank (where 1 is most deprived)
   Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs)
   Crime Score
   Crime Rank (where 1 i

In [13]:
out_file = PROCESSED_DIR / "imd2025_processed.parquet"

imd.to_parquet(out_file, index=False)

chk = pd.read_parquet(out_file)
print(f"Saved {len(chk):,} rows to {out_file} ({out_file.stat().st_size / 1e6:,.2f} MB)")
print(chk.dtypes)

Saved 33,755 rows to /Users/trinityrose/Desktop/TEST/Data/Processed/imd2025_processed.parquet (0.71 MB)
lsoa21cd        string[python]
lsoa21nm        string[python]
lad24cd         string[python]
lad24nm         string[python]
imd25_rank               Int32
imd25_decile              Int8
dtype: object


# ONS

In [14]:
import requests, zipfile, shutil, tempfile
from pathlib import Path

ITEM_ID = "6fff67d204fd4f339591ed667a6e3642"
URL = f"https://www.arcgis.com/sharing/rest/content/items/{ITEM_ID}/data"

BASE = RAW_DIR 
OUT_DIR = BASE / "ONS"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ons_path = OUT_DIR / "ONSPD_MAY_2026_UK.csv"

if ons_path.exists():
    print("Already have:", ons_path)
else:
    tmp = Path(tempfile.mkdtemp(prefix="onspd_"))
    zip_path = tmp / "onspd.zip"
    try:
        with requests.get(URL, stream=True, timeout=(30, 300),
                          headers={"User-Agent": "Mozilla/5.0"}) as r:
            r.raise_for_status()
            expected = int(r.headers.get("Content-Length", 0))
            print("Status:", r.status_code, "| Type:", r.headers.get("Content-Type"),
                  "| Expected:", round(expected / 1e6), "MB")
            with open(zip_path, "wb") as f:
                for chunk in r.iter_content(4 * 1024 * 1024):
                    f.write(chunk)

        size = zip_path.stat().st_size
        print("Downloaded:", round(size / 1e6), "MB")

        with open(zip_path, "rb") as f:
            head = f.read(300)
        if not head.startswith(b"PK"):
            raise RuntimeError(f"Not a zip. Server sent: {head!r}")
        if expected and size < expected:
            raise RuntimeError(f"Incomplete: {size} of {expected} bytes. Re-run.")

        with zipfile.ZipFile(zip_path) as z:
            main = [n for n in z.namelist()
                    if n.lower().startswith("data/") and n.lower().endswith(".csv")
                    and "multi_csv" not in n.lower()][0]
            with z.open(main) as src, open(ons_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
        print("Kept:", ons_path)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)   # deletes zip + temp folder only

Status: 200 | Type: application/zip | Expected: 247 MB
Downloaded: 247 MB
Kept: /Users/trinityrose/Desktop/TEST/Data/Raw/ONS/ONSPD_MAY_2026_UK.csv


## Process

In [15]:
ONS_COLS = ["pcds", "doterm", "ruc21ind", "sicbl26cd", "msoa21cd", "lsoa21cd"]

# Load only those columns (the only time the data is read)
onspd = pd.read_csv(ons_path, usecols=ONS_COLS, dtype=str)[ONS_COLS]
print(f"Loaded {len(onspd):,} rows")

# Datatypes
for c in ["pcds", "ruc21ind", "sicbl26cd", "msoa21cd", "lsoa21cd"]:
    onspd[c] = onspd[c].str.strip().replace("", pd.NA).astype("string")
onspd["pcds"] = onspd["pcds"].str.upper()
onspd["doterm"] = pd.to_datetime(onspd["doterm"].str.strip().replace("", pd.NA),
                                 format="%Y%m", errors="coerce")

Loaded 2,726,477 rows


In [16]:
out_file = PROCESSED_DIR / "onspd_processed.parquet"

onspd.to_parquet(out_file, index=False)

# Read it back to check
chk = pd.read_parquet(out_file)
print(f"Saved {len(chk):,} rows, {chk.shape[1]} columns to {out_file} ({out_file.stat().st_size / 1e6:,.1f} MB)")
print(chk.dtypes)

Saved 2,726,477 rows, 6 columns to /Users/trinityrose/Desktop/TEST/Data/Processed/onspd_processed.parquet (17.8 MB)
pcds         string[python]
doterm       datetime64[ns]
ruc21ind     string[python]
sicbl26cd    string[python]
msoa21cd     string[python]
lsoa21cd     string[python]
dtype: object


# GeoJSON

In [17]:
BASE = "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services"
OUT_DIR = BASE_DIR / "Data" / "Raw" / "GeoJSON"
OUT_DIR.mkdir(exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "python-requests ons-geojson-download"})

def download_geojson(service_name, out_path, layer=0, page_size=1000):
    """Page through an ArcGIS FeatureServer layer and save one GeoJSON FeatureCollection."""
    layer_url = f"{BASE}/{service_name}/FeatureServer/{layer}"
    meta = session.get(layer_url, params={"f": "json"}, timeout=60).json()
    oid_field = next(
        (f["name"] for f in meta.get("fields", []) if f["type"] == "esriFieldTypeOID"),
        "OBJECTID",
    )
    page_size = min(page_size, meta.get("maxRecordCount", page_size))
    print(f"  ID field: {oid_field}, page size: {page_size}")

    features, offset = [], 0
    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "outSR": 4326,
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "orderByFields": oid_field,
        }
        r = session.get(layer_url + "/query", params=params, timeout=120)
        r.raise_for_status()
        data = r.json()
        if "error" in data:
            raise RuntimeError(data["error"])
        batch = data.get("features", [])
        features.extend(batch)
        print(f"  {service_name}: {len(features)} features")
        if not batch:
            break
        offset += len(batch)
        if not data.get("exceededTransferLimit", False) and len(batch) < page_size:
            break
    fc = {"type": "FeatureCollection", "features": features}
    out_path = Path(out_path)
    out_path.write_text(json.dumps(fc))
    print(f"Saved {len(features)} features -> {out_path} ({out_path.stat().st_size/1e6:.1f} MB)")
    return fc

In [18]:
icb = download_geojson(
    "Integrated_Care_Boards_April_2023_EN_BSC",
    OUT_DIR / "icb_boundaries_april_2023_bsc.geojson",
)

  ID field: FID, page size: 1000
  Integrated_Care_Boards_April_2023_EN_BSC: 42 features
Saved 42 features -> /Users/trinityrose/Desktop/TEST/Data/Raw/GeoJSON/icb_boundaries_april_2023_bsc.geojson (0.9 MB)


In [19]:
catalog = session.get(BASE, params={"f": "json"}, timeout=60).json()
names = [s["name"] for s in catalog["services"] if s["type"] == "FeatureServer"]

def find(*patterns):
    return [n for n in names if all(re.search(p, n, re.I) for p in patterns)]

candidates = find(r"LSOA21", r"SICBL23", r"ICB23", r"LAD23")

assert candidates, "No matching service found"
lookup = download_geojson(
    candidates[0],
    OUT_DIR / "lsoa2021_sicbl_icb_lad_april_2023_lookup.geojson",
)

  ID field: ObjectId, page size: 1000
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 1000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 2000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 3000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 4000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 5000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 6000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 7000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 8000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 9000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 10000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 11000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 12000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 13000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 14000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 15000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 16000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 17000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 18000 features
  LSOA21_SICBL23_ICB23_LAD23_EN_LU: 19000 features
  